# Module 3 - Genie space over CVE / KEV / STIG / Assets (fixed)

In [ ]:
%run ./_config


## 1. Register helper SQL functions

In [ ]:
spark.sql(f"""
CREATE OR REPLACE FUNCTION {BASE}.lookup_cve(cve STRING)
RETURNS TABLE
COMMENT 'Lookup a single CVE by ID and return its CVSS score, description, KEV status, and date added to KEV.'
RETURN
  SELECT
    c.cve_id,
    c.description,
    c.cvss_score,
    c.cvss_severity,
    k.cveID IS NOT NULL AS in_kev,
    k.dateAdded AS kev_date_added,
    k.requiredAction AS kev_required_action
  FROM {BASE}.cves c
  LEFT JOIN {BASE}.kev_catalog k ON c.cve_id = k.cveID
  WHERE c.cve_id = lookup_cve.cve
""")


In [ ]:
spark.sql(f"""
CREATE OR REPLACE FUNCTION {BASE}.assets_for_product(vendor_name STRING, product_name STRING)
RETURNS TABLE
COMMENT 'Return all assets running a given vendor and product.'
RETURN
  SELECT asset_id, hostname, environment, owner_org, last_patched
  FROM {BASE}.affected_assets
  WHERE vendor = vendor_name AND product = product_name
""")


## 2. Provision Genie space via API

In [ ]:
from databricks.sdk import WorkspaceClient
import json, time

w = WorkspaceClient()

# 1. Pick a serverless SQL warehouse
warehouses = list(w.warehouses.list())
serverless = [wh for wh in warehouses if getattr(wh, 'enable_serverless_compute', False)] or warehouses
warehouse_id = serverless[0].id
print(f"Using warehouse: {warehouse_id}")

TABLES = [
    f"{BASE}.kev_catalog",
    f"{BASE}.cves",
    f"{BASE}.affected_assets",
    f"{BASE}.attack_techniques",
    f"{BASE}.attack_groups",
    f"{BASE}.netops_outages",
]

DISPLAY_NAME = GENIE_SPACE_NAME
DESCRIPTION = "Cyber threat intelligence over CVE, KEV, MITRE ATT&CK, and DoD asset inventory."

# 2. Look for existing space; spaces aren't listable via /api/2.0/genie/spaces (returns {}),
# so we cache the space id in _workshop_config to make this idempotent.
existing_id = cfg_get('genie_space_id')

GENIE_SPACE_ID = None
if existing_id:
    try:
        w.api_client.do('GET', f'/api/2.0/genie/spaces/{existing_id}')
        GENIE_SPACE_ID = existing_id
        print(f"Reusing cached Genie space: {GENIE_SPACE_ID}")
    except Exception:
        print(f"Cached space {existing_id} not found; will create a new one.")

if not GENIE_SPACE_ID:
    inner = {
        "version": 2,
        "data_sources": {"tables": [{"identifier": t} for t in sorted(TABLES)]},
    }
    body = {
        "warehouse_id": warehouse_id,
        "title": DISPLAY_NAME,
        "description": DESCRIPTION,
        "serialized_space": json.dumps(inner),
    }
    res = w.api_client.do('POST', '/api/2.0/genie/spaces', body=body)
    GENIE_SPACE_ID = res.get('space_id') or res.get('id')
    print(f"Created Genie space: {GENIE_SPACE_ID}")

# 3. Persist for downstream notebooks
cfg_set('genie_space_id', GENIE_SPACE_ID)
cfg_set('warehouse_id', warehouse_id)

host = w.config.host.rstrip('/')
print(f"\nGENIE_SPACE_ID = {GENIE_SPACE_ID}")
print(f"Open in UI:    {host}/genie/rooms/{GENIE_SPACE_ID}")
print()
print("Tip: the API does not yet support setting Genie instructions/sample questions")
print("     in `serialized_space`. The space is created with the 5 tables above.")
print("     Sample questions and instructions can be added via UI if desired; the")
print("     compound agent in notebook 05 still works without them.")


## 3. Save Genie SQL for Module 6 (sanity-check the query)

In [ ]:
GENIE_SQL = """
SELECT vendorProject AS vendor, COUNT(*) AS cve_count
FROM {BASE}.kev_catalog
GROUP BY vendorProject
ORDER BY cve_count DESC
LIMIT 10
"""

result = spark.sql(GENIE_SQL)
result.printSchema()
result.display()
